# scratch

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

rng = np.random.default_rng()
π = np.pi

%matplotlib ipympl
    
%load_ext autoreload
%autoreload all
    
fig = plt.figure(constrained_layout=True)

def mkax():
    fig.clf()
    return fig.gca()

In [ ]:
ax = mkax()
ax.plot(signal.get_window("hann", 50))
ax.plot(signal.get_window("hamming", 50))
ax.plot(signal.windows.kaiser(50, 5))

In [ ]:
from modreczoo.data import load_dataset_loader

dl, x, meta, label_to_id = load_dataset_loader(
    Path("data/awgn_snr0_30/"), model_name="time_cnn", channel_format="real_imag")

In [ ]:
meta[i]

In [ ]:
n = np.arange(x.shape[1])
i = 1

ax = mkax()
ax.plot(n, x[i].real, n, x[i].imag)
ax.set(title=f"{meta[i,"modulation"]}, OSR={meta[i,"osr"]}, symbols={meta[i,"n_symbols"]}, SNR={meta[i,"snr_db"]:.1f}dB");

## Understanding Entropy

In [ ]:
from scipy import stats

def cross_entropy(P, Q, x, nats=True):
    scale = 1.0 if nats else 1.0/np.log(2)
    return scale * np.trapezoid(-P*np.log(Q), x)

def entropy(P, x, nats=True):
    return cross_entropy(P, P, x, nats)

# def joint_entropy(P, Q, x, nats=True):
#     return np.trapezoid(

def kl_divergence(P, Q, x, nats=True):
    scale = 1.0 if nats else 1.0/np.log(2)
    return scale * np.trapezoid(P*np.log(P/Q), x)

relative_entropy = kl_divergence

P = stats.norm(loc=0, scale=1)
# Q = stats.norm(loc=2, scale=1)
# Q = stats.uniform()
Q = stats.norm(loc=10, scale=1)
x = np.linspace(-10, 20, 1000)
dx = x[1] - x[0]

nats = False
H_PQ = cross_entropy(P.pdf(x), Q.pdf(x), x, nats)
H_QP = cross_entropy(Q.pdf(x), P.pdf(x), x, nats)
H_P = entropy(P.pdf(x), x, nats)
H_Q = entropy(Q.pdf(x), x, nats)
D_PQ = kl_divergence(P.pdf(x), Q.pdf(x), x, nats)
D_QP = kl_divergence(Q.pdf(x), P.pdf(x), x, nats)

ax = mkax()
ax.plot(x, P.pdf(x), x, Q.pdf(x))
ax.set(title=f"{H_P=:.2f}, {H_Q=:.2f}\n{H_PQ=:.2f}, {H_QP=:.2f}\n{D_PQ=:.2f}, {D_QP=:.2f}")

np.allclose(D_PQ, H_PQ - H_P), np.allclose(D_QP, H_QP - H_Q)

## Interactive spectrogram preprocessing browser

Use this to scroll through representative examples from each modulation and compare preprocessing flags side by side. The default dataset path points at the larger high-SNR dataset from the Kaiser sweep.

In [ ]:
from pathlib import Path

from IPython.display import display
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import polars as pl

from modreczoo.data import load_dataset, normalize_signal, spectrogram_channels

DATASET_DIR = Path("data/spectrogram_high_snr_sobol_16384")
REPRESENTATIVES_PER_CLASS = 24

signals_browser, meta_browser = load_dataset(str(DATASET_DIR))
meta_indexed = meta_browser.with_row_index("_row")
modulation_labels = sorted(meta_browser["modulation"].unique().to_list())


def representative_indices(metadata: pl.DataFrame, labels: list[str], per_class: int) -> dict[str, list[int]]:
    reps: dict[str, list[int]] = {}
    for label in labels:
        rows = metadata.filter(pl.col("modulation") == label).sort(["snr_db", "osr", "signal_id"])
        if rows.is_empty():
            reps[label] = []
            continue
        positions = np.linspace(0, rows.height - 1, min(per_class, rows.height)).round().astype(int)
        reps[label] = rows[positions]["_row"].to_list()
    reps["All"] = [idx for label in labels for idx in reps[label]]
    return reps


rep_indices = representative_indices(meta_indexed, modulation_labels, REPRESENTATIVES_PER_CLASS)

WINDOW_CONFIGS = {
    "hann": {"window": "hann", "window_beta": 0.0},
    "kaiser β=0": {"window": "kaiser", "window_beta": 0.0},
    "kaiser β=5": {"window": "kaiser", "window_beta": 5.0},
    "kaiser β=15": {"window": "kaiser", "window_beta": 15.0},
}

modulation_widget = widgets.Dropdown(options=["All", *modulation_labels], value="All", description="Class")
sample_widget = widgets.IntSlider(value=0, min=0, max=len(rep_indices["All"]) - 1, step=1, description="Sample")
size_widget = widgets.Dropdown(options=[32, 64, 96, 128], value=64, description="Size")
nperseg_widget = widgets.Dropdown(options=[16, 32, 64, 128], value=64, description="nperseg")
overlap_widget = widgets.Dropdown(options=[0, 8, 16, 32, 48, 64, 96], value=48, description="noverlap")
format_widget = widgets.Dropdown(options=["mag_phase", "mag_inst_freq", "real_imag"], value="mag_phase", description="Format")
channel_widget = widgets.Dropdown(options=[0, 1], value=0, description="Channel")
config_widget = widgets.SelectMultiple(
    options=list(WINDOW_CONFIGS),
    value=("kaiser β=0", "kaiser β=5", "kaiser β=15"),
    description="Windows",
    rows=4,
)
cmap_widget = widgets.Dropdown(options=["magma", "viridis", "twilight", "RdBu_r"], value="magma", description="Cmap")


def _sync_sample_range(*_):
    current = rep_indices[modulation_widget.value]
    sample_widget.max = max(len(current) - 1, 0)
    sample_widget.value = min(sample_widget.value, sample_widget.max)


modulation_widget.observe(_sync_sample_range, names="value")


def _metadata_row(signal_idx: int) -> dict:
    return meta_browser[signal_idx].to_dicts()[0]


def _channel_name(channel_format: str, channel: int) -> str:
    names = {
        "mag_phase": ("log magnitude", "phase / π"),
        "mag_inst_freq": ("log magnitude", "instantaneous frequency"),
        "real_imag": ("real", "imag"),
    }
    return names[channel_format][channel]


def render_browser(modulation, sample, size, nperseg, noverlap, channel_format, channel, configs, cmap):
    if noverlap >= nperseg:
        print("Choose noverlap < nperseg.")
        return
    if not configs:
        print("Select at least one preprocessing config.")
        return

    choices = rep_indices[modulation]
    signal_idx = choices[sample % len(choices)]
    row = _metadata_row(signal_idx)
    x = normalize_signal(signals_browser[signal_idx])
    t = np.arange(len(x))

    fig, axes = plt.subplots(1, len(configs) + 1, figsize=(4.2 * (len(configs) + 1), 3.8), constrained_layout=True)
    axes = np.atleast_1d(axes)

    axes[0].plot(t, x.real, lw=0.8, label="I")
    axes[0].plot(t, x.imag, lw=0.8, label="Q")
    axes[0].set_title(
        f"#{signal_idx} {row['modulation']}\n"
        f"SNR={row['snr_db']:.1f} dB OSR={row['osr']} EBW={row['ebw']:.2f}"
    )
    axes[0].set_xlabel("sample")
    axes[0].legend(loc="upper right", fontsize=8)
    axes[0].grid(alpha=0.2)

    for ax, label in zip(axes[1:], configs):
        cfg = WINDOW_CONFIGS[label]
        features = spectrogram_channels(
            x,
            channel_format=channel_format,
            size=size,
            nperseg=nperseg,
            noverlap=noverlap,
            window=cfg["window"],
            window_beta=cfg["window_beta"],
        )
        image = features[channel]
        im = ax.imshow(image, origin="lower", aspect="auto", cmap=cmap)
        ax.set_title(f"{label}\n{_channel_name(channel_format, channel)}")
        ax.set_xlabel("time bin")
        ax.set_ylabel("frequency bin")
        fig.colorbar(im, ax=ax, shrink=0.75)

    display(fig)
    plt.close(fig)


controls = widgets.VBox([
    widgets.HBox([modulation_widget, sample_widget]),
    widgets.HBox([size_widget, nperseg_widget, overlap_widget]),
    widgets.HBox([format_widget, channel_widget, cmap_widget]),
    config_widget,
])
out = widgets.interactive_output(
    render_browser,
    {
        "modulation": modulation_widget,
        "sample": sample_widget,
        "size": size_widget,
        "nperseg": nperseg_widget,
        "noverlap": overlap_widget,
        "channel_format": format_widget,
        "channel": channel_widget,
        "configs": config_widget,
        "cmap": cmap_widget,
    },
)
display(controls, out)

In [ ]:
!uv run modreczoo-train --dataset-dir data/awgn_snr0_30 --models time_cnn --epochs 1 --max-examples 256 --batch-size 64 --num-workers 0 --device cpu